# AZ80 Microstructure-to-Properties: From-Scratch Training

Trains `Net_conditional` (image -> YS/UTS/EL/E/k/n) from randomly initialized weights.

Built from the two source files in this repo:
- `microstructure-to-properties.ipynb` -- source of truth for the model architecture, EMA class, `save_model()`/`load_model()`, `class_maker()`, `noise()`, and the `test()` evaluation function.
- `train.py` -- the project's actual training script (this is a *continuation* run: it restores an 800-epoch checkpoint and trains 200 more epochs on top of it).

Unlike `train.py`, this notebook trains from scratch: the pretrained-checkpoint `load_model()` call before the loop is **not** made, `n_epoch` is set to ~800 (from the paper, Section 3.2) instead of 200, and early stopping is added on the combined seen+unseen validation loss.

**Live visualization:** while training, this notebook redraws a loss-curve plot (train/seen-val/unseen-val total loss) every epoch, and every `image_log_every` epochs (default 10, plus epoch 1, plus the run's final epoch) shows a grid of sample validation images next to the model's predicted vs. true property values and their % error -- see the "Visualization helpers" cell below. This is notebook-only; `train_from_scratch.py` does not include it.

**Paper-style diagnostics:** at that same cadence, the notebook also shows three evaluations modeled directly on the source paper's own figures -- Grad-CAM heatmaps (which image regions the model relies on), a per-class predicted-vs-target scatter with NRMSE/R² (predictions averaged over every image of a class, seen vs. unseen), and reconstructed Ramberg-Osgood stress-strain curves from real vs. predicted E/K/n -- see the "Advanced diagnostics (paper-style)" cell below.

**Training data status:** the paper's real `Train-Oversampled` dataset (83 classes, synthetically oversampled via a diffusion model) is not available in this repo. `az80-microstructure-data/` currently only has `Test-Seen`/`Test-Unseen` (72 images, 46 classes total). If `Train-Oversampled` is missing, this notebook automatically falls back to a small merged training set built from `Test-Seen`+`Test-Unseen` so the pipeline can still be run end-to-end -- **this fallback does not produce a meaningful model** (see the "Training data fallback" cell below for why). Add the real `Train-Oversampled` folder to disable it.

See `README.md` in this repo for exactly which values are ported verbatim vs. newly introduced.

**Bootstrap:** the next cell clones this repo (if the dataset isn't already reachable from the current working directory), `cd`s into it, and installs `requirements.txt` -- run it first, before the Imports cell. Works unmodified on Colab, Kaggle, or a fresh local Jupyter checkout; it's a no-op if the dataset is already present (e.g. you're already running from inside a clone of this repo). Enable a GPU runtime/accelerator first (Colab: Runtime -> Change runtime type; Kaggle: Settings -> Accelerator).


In [ ]:
import os
import subprocess
import sys

# new -- bootstrap, generalized to any platform (Colab, Kaggle, a fresh local
# Jupyter, ...). Originally this only cloned when "google.colab" in sys.modules,
# which meant a non-Colab environment (e.g. a fresh Kaggle notebook, whose
# tracebacks look like /tmp/ipykernel_NNN/....py rather than Colab's
# <ipython-input-N-hash>) silently skipped the clone and then failed later with
# FileNotFoundError on az80-microstructure-data/Test-Seen. Instead, check directly
# for the dataset: clone+cd whenever it isn't already reachable from cwd, and skip
# entirely (no-op) when it already is (e.g. running from an existing local checkout).
DATASET_MARKER = os.path.join("az80-microstructure-data", "Test-Seen")

if not os.path.isdir(DATASET_MARKER):
    repo_dir = "/content/phase2" if "google.colab" in sys.modules else os.path.abspath("phase2")
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(
            ["git", "clone", "https://github.com/arhorri/phase2.git", repo_dir],
            check=True)
    os.chdir(repo_dir)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True)
    print(f"Bootstrap done -- cloned/entered repo, cwd: {os.getcwd()}")
else:
    print(f"Dataset already reachable at '{DATASET_MARKER}' -- skipping clone.")


## Imports

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import os
import copy
import csv
import shutil
import matplotlib.pyplot as plt
from IPython.display import clear_output

## Config
*new* -- all paths/hyperparameters that must be edited per-environment are collected here (matching the editable-variable convention already used for `external_validation_dir` in `microstructure-to-properties.ipynb`).

The dataset directories default to the copy committed in this repo (`az80-microstructure-data/...`). Override `WHOLE_DIR`/`SEEN_TEST_DIR`/`UNSEEN_TEST_DIR`/`SAVE_DIR`/`RESUME_DIR` as environment variables only if you keep the data somewhere else.

`SEEN_TEST_DIR`/`UNSEEN_TEST_DIR` are checked here and raise immediately if missing. `WHOLE_DIR` (the training set) is checked separately, further down -- see the next cell.

In [ ]:
trial = 1

# --- Data / checkpoint directories --------------------------------------------------
# new -- these were hardcoded Kaggle absolute paths in train.py
# (e.g. '/kaggle/input/mic-mech/Mic-Mech/Train-Oversampled'). The dataset now ships
# inside this repo under az80-microstructure-data/, so the defaults below work
# unchanged locally, in Colab, or after `git clone` on Kaggle -- no per-environment
# dataset attachment needed. Override via env vars only if you keep the data
# somewhere else (e.g. a separate Kaggle Dataset mounted at /kaggle/input/...).
whole_dir = os.environ.get("WHOLE_DIR", "az80-microstructure-data/Train-Oversampled")
seen_test_dir = os.environ.get("SEEN_TEST_DIR", "az80-microstructure-data/Test-Seen")
unseen_test_dir = os.environ.get("UNSEEN_TEST_DIR", "az80-microstructure-data/Test-Unseen")
save_dir = os.environ.get("SAVE_DIR", "checkpoints/")
os.makedirs(save_dir, exist_ok=True)

# new -- fail with a clear message instead of torchvision's raw FileNotFoundError
# from deep inside ImageFolder if a directory is missing. WHOLE_DIR is checked
# separately, further down, where a missing Train-Oversampled folder triggers an
# interim fallback instead of a hard failure -- see "interim training-data fallback"
# below.
for _name, _path in [("SEEN_TEST_DIR", seen_test_dir), ("UNSEEN_TEST_DIR", unseen_test_dir)]:
    if not os.path.isdir(_path):
        raise FileNotFoundError(
            f"{_name} does not exist: '{_path}'. Set the {_name} environment "
            f"variable to the correct path, or add the missing folder under "
            f"az80-microstructure-data/.")

# new -- train.py loaded a pretrained checkpoint here before training (making it a
# continuation/fine-tuning run). This script trains from scratch, so no checkpoint is
# loaded at start. `resume_dir` is kept only as an optional manual-resume hook: leave
# it as None for a genuine from-scratch run.
resume_dir = os.environ.get("RESUME_DIR", None)

# ported from train.py -- auto cuda/cpu detection, used unchanged
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Hyperparameters -----------------------------------------------------------------
# ported from train.py -- exact values used in practice, do not change without reason
batch_size = 4              # train.py: batch_size = 4 (kept configurable for smaller/larger GPUs)
learning_rate = 3e-5         # train.py: learning_rate = 3e-5
image_shape = (1, 512, 512)  # train.py: image_shape
image_size = 512             # train.py: image_size
magnifications = 4           # train.py: magnifications
embedding_dim = 100          # train.py: embedding_dim
num_classes = 83             # train.py: num_classes
ys_range = 105
uts_range = 125
el_range = 9.25
e_range = 12
k_range = 250.11
n_range = 0.104

# new -- added to convert continuation run into from-scratch training.
# The paper (Section 3.2) states training "was completed after approximately 800
# epochs" with early stopping -- that is the basis for n_epoch below. train.py itself
# only ever ran 200-epoch *continuation* windows on top of an already-trained
# checkpoint, so it has no single "total epochs from scratch" value to port.
n_epoch = 800          # from the paper (~800 epochs to convergence), NOT from train.py
patience = 50          # new -- our own default; not specified in the paper or train.py
min_delta = 0.0        # new -- our own default; minimum improvement to reset patience

# new -- train.py initialized min_loss = 1848.41, a value carried over from a *prior*
# training run's checkpoint. A from-scratch run has no such prior baseline, so we start
# from +inf, matching microstructure-to-properties.ipynb / train.py's own
# `total_loss_min = np.Inf` convention (spelled np.inf here -- np.Inf was removed in
# NumPy 2.0, which Kaggle's images now ship).
min_loss = np.inf

# new -- how often (in epochs) to display a grid of sample validation images
# with predicted-vs-true property values, in addition to the per-epoch loss plot
# (which updates every epoch regardless). Image grids are heavier to render, so
# they default to every 10th epoch rather than every epoch.
image_log_every = int(os.environ.get("IMAGE_LOG_EVERY", 10))

## Class / property lookup tables
*ported from train.py* -- `class_table` / `seen_table` / `unseen_table`, used to look up YS/UTS/EL/E/K/N for each `ImageFolder` class index. Copied verbatim.

In [ ]:
class_table = torch.tensor([[    0.000,     1.000,     2.000,     3.000,     0.000,     1.000,
             2.000,     3.000,     0.000,     1.000,     2.000,     3.000,
             0.000,     1.000,     3.000,     0.000,     1.000,     2.000,
             3.000,     0.000,     1.000,     2.000,     3.000,     0.000,
             2.000,     0.000,     1.000,     2.000,     3.000,     0.000,
             1.000,     3.000,     0.000,     1.000,     3.000,     0.000,
             1.000,     2.000,     3.000,     0.000,     1.000,     2.000,
             3.000,     0.000,     1.000,     3.000,     0.000,     1.000,
             2.000,     3.000,     0.000,     1.000,     2.000,     3.000,
             0.000,     1.000,     3.000,     0.000,     1.000,     3.000,
             0.000,     1.000,     3.000,     0.000,     1.000,     3.000,
             0.000,     1.000,     2.000,     3.000,     0.000,     1.000,
             2.000,     3.000,     0.000,     1.000,     3.000,     0.000,
             1.000,     3.000,     0.000,     1.000,     3.000],
        [  206.000,   206.000,   206.000,   206.000,   193.000,   193.000,
           193.000,   193.000,   212.000,   212.000,   212.000,   212.000,
           251.000,   251.000,   251.000,   244.000,   244.000,   244.000,
           244.000,   151.000,   151.000,   151.000,   151.000,   148.000,
           148.000,   178.000,   178.000,   178.000,   178.000,   210.000,
           210.000,   210.000,   195.000,   195.000,   195.000,   177.000,
           177.000,   177.000,   177.000,   146.000,   146.000,   146.000,
           146.000,   183.000,   183.000,   183.000,   202.000,   202.000,
           202.000,   202.000,   172.000,   172.000,   172.000,   172.000,
           216.000,   216.000,   216.000,   197.000,   197.000,   197.000,
           227.000,   227.000,   227.000,   230.000,   230.000,   230.000,
           223.000,   223.000,   223.000,   223.000,   198.000,   198.000,
           198.000,   198.000,   183.000,   183.000,   183.000,   216.000,
           216.000,   216.000,   209.000,   209.000,   209.000],
        [  293.000,   293.000,   293.000,   293.000,   284.000,   284.000,
           284.000,   284.000,   289.000,   289.000,   289.000,   289.000,
           330.000,   330.000,   330.000,   309.000,   309.000,   309.000,
           309.000,   230.000,   230.000,   230.000,   230.000,   210.000,
           210.000,   226.000,   226.000,   226.000,   226.000,   271.000,
           271.000,   271.000,   239.000,   239.000,   239.000,   299.000,
           299.000,   299.000,   299.000,   256.000,   256.000,   256.000,
           256.000,   252.000,   252.000,   252.000,   280.000,   280.000,
           280.000,   280.000,   246.000,   246.000,   246.000,   246.000,
           335.000,   335.000,   335.000,   306.000,   306.000,   306.000,
           311.000,   311.000,   311.000,   305.000,   305.000,   305.000,
           302.000,   302.000,   302.000,   302.000,   326.000,   326.000,
           326.000,   326.000,   299.000,   299.000,   299.000,   307.000,
           307.000,   307.000,   305.000,   305.000,   305.000],
        [    5.830,     5.830,     5.830,     5.830,     5.750,     5.750,
             5.750,     5.750,     4.160,     4.160,     4.160,     4.160,
             4.980,     4.980,     4.980,     3.170,     3.170,     3.170,
             3.170,     3.520,     3.520,     3.520,     3.520,     3.490,
             3.490,     2.130,     2.130,     2.130,     2.130,     2.640,
             2.640,     2.640,     1.590,     1.590,     1.590,     4.790,
             4.790,     4.790,     4.790,     5.320,     5.320,     5.320,
             5.320,     2.220,     2.220,     2.220,     4.210,     4.210,
             4.210,     4.210,     1.900,     1.900,     1.900,     1.900,
             6.900,     6.900,     6.900,    10.010,    10.010,    10.010,
             6.310,     6.310,     6.310,     4.670,     4.670,     4.670,
             5.710,     5.710,     5.710,     5.710,     6.390,     6.390,
             6.390,     6.390,    10.840,    10.840,    10.840,     7.240,
             7.240,     7.240,     7.890,     7.890,     7.890],
        [   45.000,    45.000,    45.000,    45.000,    46.000,    46.000,
            46.000,    46.000,    46.000,    46.000,    46.000,    46.000,
            42.000,    42.000,    42.000,    45.000,    45.000,    45.000,
            45.000,    43.000,    43.000,    43.000,    43.000,    37.000,
            37.000,    38.000,    38.000,    38.000,    38.000,    42.000,
            42.000,    42.000,    42.000,    42.000,    42.000,    47.000,
            47.000,    47.000,    47.000,    47.000,    47.000,    47.000,
            47.000,    47.000,    47.000,    47.000,    40.000,    40.000,
            40.000,    40.000,    45.000,    45.000,    45.000,    45.000,
            49.000,    49.000,    49.000,    46.000,    46.000,    46.000,
            41.000,    41.000,    41.000,    44.000,    44.000,    44.000,
            45.000,    45.000,    45.000,    45.000,    47.000,    47.000,
            47.000,    47.000,    46.000,    46.000,    46.000,    45.000,
            45.000,    45.000,    43.000,    43.000,    43.000],
        [  445.050,   445.050,   445.050,   445.050,   444.060,   444.060,
           444.060,   444.060,   450.870,   450.870,   450.870,   450.870,
           471.470,   471.470,   471.470,   462.380,   462.380,   462.380,
           462.380,   429.330,   429.330,   429.330,   429.330,   365.450,
           365.450,   408.530,   408.530,   408.530,   408.530,   447.580,
           447.580,   447.580,   490.510,   490.510,   490.510,   552.210,
           552.210,   552.210,   552.210,   470.790,   470.790,   470.790,
           470.790,   523.370,   523.370,   523.370,   450.070,   450.070,
           450.070,   450.070,   615.560,   615.560,   615.560,   615.560,
           540.690,   540.690,   540.690,   458.930,   458.930,   458.930,
           450.830,   450.830,   450.830,   444.030,   444.030,   444.030,
           442.340,   442.340,   442.340,   442.340,   553.940,   553.940,
           553.940,   553.940,   454.920,   454.920,   454.920,   448.660,
           448.660,   448.660,   461.020,   461.020,   461.020],
        [    0.123,     0.123,     0.123,     0.123,     0.130,     0.130,
             0.130,     0.130,     0.117,     0.117,     0.117,     0.117,
             0.092,     0.092,     0.092,     0.096,     0.096,     0.096,
             0.096,     0.161,     0.161,     0.161,     0.161,     0.143,
             0.143,     0.130,     0.130,     0.130,     0.130,     0.116,
             0.116,     0.116,     0.147,     0.147,     0.147,     0.175,
             0.175,     0.175,     0.175,     0.181,     0.181,     0.181,
             0.181,     0.167,     0.167,     0.167,     0.122,     0.122,
             0.122,     0.122,     0.196,     0.196,     0.196,     0.196,
             0.148,     0.148,     0.148,     0.132,     0.132,     0.132,
             0.102,     0.102,     0.102,     0.098,     0.098,     0.098,
             0.103,     0.103,     0.103,     0.103,     0.162,     0.162,
             0.162,     0.162,     0.140,     0.140,     0.140,     0.110,
             0.110,     0.110,     0.119,     0.119,     0.119]])

seen_table = torch.tensor([[    0.000,     0.000,     1.000,     1.000,     3.000,     1.000,
             2.000,     0.000,     1.000,     2.000,     3.000,     0.000,
             1.000,     0.000,     1.000,     0.000,     1.000,     0.000,
             0.000,     1.000,     0.000,     1.000,     3.000,     0.000,
             1.000,     3.000,     0.000,     1.000,     3.000,     0.000,
             1.000,     3.000],
        [  193.000,   212.000,   212.000,   244.000,   244.000,   151.000,
           148.000,   178.000,   178.000,   178.000,   178.000,   177.000,
           177.000,   183.000,   183.000,   202.000,   202.000,   216.000,
           197.000,   197.000,   227.000,   227.000,   227.000,   230.000,
           230.000,   230.000,   183.000,   183.000,   183.000,   216.000,
           216.000,   216.000],
        [  284.000,   289.000,   289.000,   309.000,   309.000,   230.000,
           210.000,   226.000,   226.000,   226.000,   226.000,   299.000,
           299.000,   252.000,   252.000,   280.000,   280.000,   335.000,
           306.000,   306.000,   311.000,   311.000,   311.000,   305.000,
           305.000,   305.000,   299.000,   299.000,   299.000,   307.000,
           307.000,   307.000],
        [    5.750,     4.160,     4.160,     3.170,     3.170,     3.520,
             3.490,     2.130,     2.130,     2.130,     2.130,     4.790,
             4.790,     2.220,     2.220,     4.210,     4.210,     6.900,
            10.010,    10.010,     6.310,     6.310,     6.310,     4.670,
             4.670,     4.670,    10.840,    10.840,    10.840,     7.240,
             7.240,     7.240],
        [   46.000,    46.000,    46.000,    45.000,    45.000,    43.000,
            37.000,    38.000,    38.000,    38.000,    38.000,    47.000,
            47.000,    47.000,    47.000,    40.000,    40.000,    49.000,
            46.000,    46.000,    41.000,    41.000,    41.000,    44.000,
            44.000,    44.000,    46.000,    46.000,    46.000,    45.000,
            45.000,    45.000],
        [  444.060,   450.870,   450.870,   462.380,   462.380,   429.330,
           365.450,   408.530,   408.530,   408.530,   408.530,   552.210,
           552.210,   523.370,   523.370,   450.070,   450.070,   540.690,
           458.930,   458.930,   450.830,   450.830,   450.830,   444.030,
           444.030,   444.030,   454.920,   454.920,   454.920,   448.660,
           448.660,   448.660],
        [    0.130,     0.117,     0.117,     0.096,     0.096,     0.161,
             0.143,     0.130,     0.130,     0.130,     0.130,     0.175,
             0.175,     0.167,     0.167,     0.122,     0.122,     0.148,
             0.132,     0.132,     0.102,     0.102,     0.102,     0.098,
             0.098,     0.098,     0.140,     0.140,     0.140,     0.110,
             0.110,     0.110]])

unseen_table = torch.tensor([[    0.000,     1.000,     2.000,     3.000,     1.000,     0.000,
             1.000,     2.000,     3.000,     2.000,     0.000,     1.000,
             2.000,     3.000],
        [  223.000,   223.000,   223.000,   223.000,   148.000,   198.000,
           198.000,   198.000,   198.000,   216.000,   208.000,   208.000,
           208.000,   208.000],
        [  302.000,   302.000,   302.000,   302.000,   210.000,   258.000,
           258.000,   258.000,   258.000,   335.000,   311.000,   311.000,
           311.000,   311.000],
        [    4.280,     4.280,     4.280,     4.280,     3.490,     2.750,
             2.750,     2.750,     2.750,     6.900,     6.370,     6.370,
             6.370,     6.370],
        [   43.000,    43.000,    43.000,    43.000,    37.000,    42.000,
            42.000,    42.000,    42.000,    49.000,    38.000,    38.000,
            38.000,    38.000],
        [  466.380,   466.380,   466.380,   466.380,   365.450,   428.760,
           428.760,   428.760,   428.760,   540.690,   485.180,   485.180,
           485.180,   485.180],
        [    0.115,     0.115,     0.115,     0.115,     0.143,     0.120,
             0.120,     0.120,     0.120,     0.148,     0.125,     0.125,
             0.125,     0.125]])

## Training data fallback
*ported from train.py* -- the `seen_label`/`unseen_label` name dictionaries (previously unused by the training loop itself, only needed now for the fallback below).

*new -- added to convert continuation run into from-scratch training, and to cope with the real training set being unavailable*: if `WHOLE_DIR` (`Train-Oversampled`) doesn't exist, this cell auto-builds a small merged training set from `Test-Seen`+`Test-Unseen` instead, holding out one image per class where more than one is available, so the pipeline (forward/backward pass, EMA, checkpointing, CSV export) can be exercised end-to-end.

**This is not the paper's real training data.** Validation below still reads the *original* `Test-Seen`/`Test-Unseen` folders, so most images end up in both training and validation under this fallback -- it does not measure generalization and results from it are not scientifically meaningful. It exists purely to prove the code runs correctly.

Once the real `Train-Oversampled` folder is added, `WHOLE_DIR` will exist, this fallback is skipped automatically, and the original 83-class `class_table` is used exactly as `train.py` intended -- no cells need to change.

In [ ]:
# ported from train.py -- label-string dictionaries for seen_table/unseen_table
seen_label = {
    0: "CM04-0500", 1: "CM10-0500", 2: "CM10-1000", 3: "CM16-1000",
    4: "CM16-2000", 5: "CS01-1000", 6: "CS04-1500", 7: "CS10-0500",
    8: "CS10-1000", 9: "CS10-1500", 10: "CS10-2000", 11: "PL03-0500",
    12: "PL03-1000", 13: "PL13-0500", 14: "PL13-1000", 15: "PL16-0500",
    16: "PL16-1000", 17: "PM03-0500", 18: "PM11-0500", 19: "PM11-1000",
    20: "PM13-0500", 21: "PM13-1000", 22: "PM13-2000", 23: "PM16-0500",
    24: "PM16-1000", 25: "PM16-2000", 26: "PS11-0500", 27: "PS11-1000",
    28: "PS11-2000", 29: "PS16-0500", 30: "PS16-1000", 31: "PS16-2000"
}

unseen_label = {
    0: 'CM06-0500', 1: 'CM06-1000', 2: 'CM06-1500', 3: 'CM06-2000',
    4: 'CS04-1000', 5: 'CS06-0500', 6: 'CS06-1000', 7: 'CS06-1500',
    8: 'CS06-2000', 9: 'PM03-1500', 10: 'PS13-0500', 11: 'PS13-1000',
    12: 'PS13-1500', 13: 'PS13-2000'
}

# =====================================================================================
# new -- interim training-data fallback. The paper's real Train-Oversampled dataset
# (83 classes, oversampled via a diffusion model) is not available in this repo.
# If WHOLE_DIR doesn't exist, auto-build a small merged training set from Test-Seen +
# Test-Unseen instead, so the pipeline (forward/backward pass, EMA, checkpointing, CSV
# export) can be exercised end-to-end while the real data is unavailable.
#
# IMPORTANT: validation below still reads the ORIGINAL Test-Seen/Test-Unseen folders,
# so most images end up in both training and validation under this fallback -- it does
# NOT measure generalization and its results are not scientifically meaningful. This
# exists purely to prove the code runs. See README.md "Training data status".
#
# Once the real Train-Oversampled folder is added, WHOLE_DIR will exist, this whole
# block is skipped automatically, and the original 83-class `class_table` is used
# exactly as train.py intended -- no config changes needed.
# =====================================================================================

def _name_to_props(table, label_dict):
    return {name: table[:, idx] for idx, name in label_dict.items()}


_known_props = {**_name_to_props(seen_table, seen_label), **_name_to_props(unseen_table, unseen_label)}


def _build_class_table(directory, name_to_props):
    class_names = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
    missing = [n for n in class_names if n not in name_to_props]
    if missing:
        raise ValueError(f"No known property values for classes {missing} in '{directory}'")
    return torch.stack([name_to_props[n] for n in class_names], dim=1)


def _build_fallback_train_dir(dest_dir, *source_dirs):
    if os.path.isdir(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)
    n_images = 0
    n_classes = 0
    for src in source_dirs:
        for entry in sorted(os.scandir(src), key=lambda e: e.name):
            if not entry.is_dir():
                continue
            files = sorted(os.listdir(entry.path))
            if not files:
                continue
            # keep all but one image per class for training; classes with only one
            # image contribute it to training too (there is nothing left to hold out)
            keep = files[:-1] if len(files) > 1 else files
            dst_class_dir = os.path.join(dest_dir, entry.name)
            os.makedirs(dst_class_dir, exist_ok=True)
            for fname in keep:
                shutil.copy2(os.path.join(entry.path, fname), os.path.join(dst_class_dir, fname))
            n_images += len(keep)
            n_classes += 1
    return n_images, n_classes


using_fallback_train_data = not os.path.isdir(whole_dir)
if using_fallback_train_data:
    _fallback_dir = os.path.join(os.path.dirname(os.path.normpath(seen_test_dir)), "_generated_train_small")
    _n_images, _n_classes = _build_fallback_train_dir(_fallback_dir, seen_test_dir, unseen_test_dir)
    print(f"WARNING: '{whole_dir}' not found -- using an auto-built {_n_images}-image, "
          f"{_n_classes}-class training set from '{seen_test_dir}' + '{unseen_test_dir}' "
          f"instead. This is NOT the paper's real training data and will not produce a "
          f"meaningful model -- it only exercises the pipeline end-to-end. See README.md.")
    whole_dir = _fallback_dir
    training_class_table = _build_class_table(whole_dir, _known_props)
else:
    training_class_table = class_table

## Data pipeline
*ported from train.py* -- transforms, datasets, loaders. The `whole_transform` pipeline order (Grayscale -> RandomCrop(512) -> ColorJitter -> ToTensor -> Normalize) and the separate per-batch `aug_transform` are copied verbatim. `train_dataset` now reads from `whole_dir`, which may have been redirected to the fallback directory by the previous cell.

In [ ]:
whole_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.RandomCrop(512),
    transforms.ColorJitter(brightness=0.5, contrast=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

test_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.RandomCrop(512),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
])

train_dataset = datasets.ImageFolder(whole_dir, transform=whole_transform)
seen_test_dataset = datasets.ImageFolder(seen_test_dir, transform=test_transform)
unseen_test_dataset = datasets.ImageFolder(unseen_test_dir, transform=test_transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
seen_test_loader = torch.utils.data.DataLoader(dataset=seen_test_dataset, batch_size=39, shuffle=True)
unseen_test_loader = torch.utils.data.DataLoader(dataset=unseen_test_dataset, batch_size=33, shuffle=True)

l = len(train_loader)

## Helper functions
*ported from microstructure-to-properties.ipynb / train.py* -- checkpoint I/O (kept in the exact `"model_state"`/`"ema_model_state"`/`"model_optimizer"` dict shape so checkpoints stay loadable by the original notebook), `class_maker()`, `noise()` (label-noise augmentation), `test()` evaluation, weight-init helpers, and the `EMA` class.

In [ ]:
def save_model(address):
    checkpoint = {"model_state": model.state_dict(),
                  "ema_model_state": ema_model.state_dict(),
                  "model_optimizer": optimizer.state_dict()}
    torch.save(checkpoint, address)


def load_model(address):
    checkpoint = torch.load(address)
    model.load_state_dict(checkpoint["model_state"])
    ema_model.load_state_dict(checkpoint["ema_model_state"])
    optimizer.load_state_dict(checkpoint["model_optimizer"])


# ported from microstructure-to-properties.ipynb / train.py -- class label unpacking
def class_maker(batch_size, labels, tab):
    Mag = torch.zeros([batch_size])
    YS = torch.zeros([batch_size])
    UTS = torch.zeros([batch_size])
    EL = torch.zeros([batch_size])
    E = torch.zeros([batch_size])
    K = torch.zeros([batch_size])
    N = torch.zeros([batch_size])

    for i in range(batch_size):
        Mag[i] = tab[0, labels[i]]
        YS[i] = tab[1, labels[i]]
        UTS[i] = tab[2, labels[i]]
        EL[i] = tab[3, labels[i]]
        E[i] = tab[4, labels[i]]
        K[i] = tab[5, labels[i]]
        N[i] = tab[6, labels[i]]

    return Mag, YS, UTS, EL, E, K, N


# ported from train.py -- label-noise augmentation applied to YS/UTS/EL/E/K/N targets
def noise(real, range_value):
    for i in range(len(real)):
        random_value = torch.randint(-10, 10, (1,))
        if random_value in range(-1, 1):
            random_noise = torch.randn(1)
            real[i] = real[i] + ((0.3 * range_value) ** 0.5) * random_noise.item()
    return real


# ported from microstructure-to-properties.ipynb -- test() evaluation function
def test(test_data, table):
    test_images, test_labels = next(iter(test_data))
    test_images = aug_transform(test_images)
    test_images = test_images.to(device)
    t_mag, t_ys, t_uts, t_el, t_e, t_k, t_n = class_maker(
        batch_size=test_labels.size(0), labels=test_labels, tab=table)

    t_mag = t_mag.long().to(device)
    t_ys = t_ys.float().to(device)
    t_uts = t_uts.float().to(device)
    t_el = t_el.float().to(device)
    t_e = t_e.float().to(device)
    t_k = t_k.float().to(device)
    t_n = t_n.float().to(device)
    test_predictions = ema_model(test_images, t_mag)
    test_ys_error = mse(t_ys, test_predictions[:, 0])
    test_uts_error = mse(t_uts, test_predictions[:, 1])
    test_el_error = mse(t_el, test_predictions[:, 2])
    test_e_error = mse(t_e, test_predictions[:, 3])
    test_k_error = mse(t_k, test_predictions[:, 4])
    test_n_error = mse(t_n, test_predictions[:, 5])
    test_total_error = (test_ys_error + test_uts_error + test_el_error
                         + test_e_error + test_k_error + test_n_error)

    return (test_ys_error, test_uts_error, test_el_error, test_e_error,
            test_k_error, test_n_error, test_total_error)


# ported from microstructure-to-properties.ipynb / train.py -- weight init helpers
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
    elif classname.find('Norm') != -1:
        torch.nn.init.normal_(m.weight, 1.0, 0.02)
        torch.nn.init.zeros_(m.bias)


def initialize_weights(self):
    for m in self.modules():
        if isinstance(m, nn.Linear):
            nnn = m.in_features
            y = 1.0 / np.sqrt(nnn)
            m.weight.data.uniform_(-y, y)
            m.bias.data.fill_(0)


# ported from microstructure-to-properties.ipynb -- EMA class
class EMA:
    def __init__(self, beta):
        super().__init__()
        self.beta = beta
        self.step = 0

    def update_model_average(self, ma_model, current_model):
        for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
            old_weight, up_weight = ma_params.data, current_params.data
            ma_params.data = self.update_average(old_weight, up_weight)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

    def step_ema(self, ema_model, model, step_start_ema=0):
        if self.step < step_start_ema:
            self.reset_parameters(ema_model, model)
            self.step += 1
            return
        if self.step == step_start_ema:
            print('EMA started!')
        self.update_model_average(ema_model, model)
        self.step += 1

    def reset_parameters(self, ema_model, model):
        ema_model.load_state_dict(model.state_dict())

## Model architecture
*ported from microstructure-to-properties.ipynb* -- `SelfAttention`, `DoubleConv`, `Down`, `Dense`, `Net_conditional`, copied verbatim. Note the folder-class count (83 normally, 46 under the fallback) has no effect on this architecture -- the model only ever consumes a magnification index (0-3) via `Down`'s embedding, not a full per-class embedding.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, channels, size):
        super(SelfAttention, self).__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 4, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels)
        )

    def forward(self, x):
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, out_channels)
        )

    def forward(self, x):
        if self.residual:
            return F.gelu(x + self.double_conv(x))
        else:
            return self.double_conv(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels, imsize, emb_dim=512):
        super().__init__()
        self.imsize = imsize
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, (out_channels - 1))
        )

        self.Mag_label = nn.Sequential(
            nn.Embedding(magnifications, embedding_dim),
            nn.Linear(embedding_dim, 1 * self.imsize * self.imsize))

    def forward(self, x, Mag):
        x = self.maxpool_conv(x)
        magemb = self.Mag_label(Mag).view(x.shape[0], 1, x.shape[-2], x.shape[-1])
        x = torch.cat((x, magemb), dim=1)
        return x


class Dense(nn.Module):
    def __init__(self, in_channels, mid_channels):
        super().__init__()
        self.linear = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, mid_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.25),
            nn.Linear(mid_channels, 6),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        x = self.linear(x)
        return x


class Net_conditional(nn.Module):
    def __init__(self, c_in=1, num_classes=None):
        super().__init__()
        self.inc = DoubleConv(c_in, 16)
        self.down1 = Down(16, 32, 256)
        self.sa1 = SelfAttention(32, 256)
        self.down2 = Down(32, 64, 128)
        self.sa2 = SelfAttention(64, 128)
        self.down3 = Down(64, 128, 64)
        self.sa3 = SelfAttention(128, 64)
        self.down4 = Down(128, 256, 32)
        self.sa4 = SelfAttention(256, 32)
        self.down5 = Down(256, 512, 16)
        self.sa5 = SelfAttention(512, 16)
        self.down6 = Down(512, 512, 8)
        self.sa6 = SelfAttention(512, 8)

        self.bot1 = DoubleConv(512, 256)
        self.bot2 = DoubleConv(256, 128)
        self.bot3 = DoubleConv(128, 64)

        self.linear = Dense(64 * 8 * 8, 64 * 8 * 8 * 2)

    def forward(self, x, Mag):
        x = self.inc(x)
        x = self.down1(x, Mag)
        x = self.down2(x, Mag)
        x = self.down3(x, Mag)
        x = self.down4(x, Mag)
        x = self.sa4(x)
        x = self.down5(x, Mag)
        x = self.sa5(x)
        x = self.down6(x, Mag)
        x = self.sa6(x)

        x = self.bot1(x)
        x = self.bot2(x)
        x = self.bot3(x)

        x = self.linear(x)
        return x

## Model / optimizer / EMA setup
*ported from train.py*, with one deliberate change: `train.py` called `load_model('.../Mic-Mech-Over-800.pth.tar')` here, restoring an already-trained checkpoint right before the training loop -- that makes it fine-tuning, not from-scratch training. That call is **not** made here. `model.apply(initialize_weights)` is kept so initialization stays explicit and reproducible. `resume_dir` is left as a manual opt-in only, for restarting an interrupted from-scratch run on Kaggle's ephemeral sessions.

In [ ]:
model = Net_conditional(num_classes=num_classes).to(device)
model.apply(initialize_weights)  # ported from train.py -- explicit, reproducible init
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
ema = EMA(0.995)
ema_model = copy.deepcopy(model).eval().requires_grad_(False)

# new -- added to convert continuation run into from-scratch training.
# train.py had `load_dir = '.../Mic-Mech-Over-800.pth.tar'; load_model(load_dir)` here,
# which restored an already-trained checkpoint right before the training loop -- that
# turns the run into fine-tuning, not training from scratch. That call is intentionally
# NOT made. `resume_dir` above is left as a manual opt-in only, for restarting an
# interrupted from-scratch run on Kaggle's ephemeral sessions.
if resume_dir is not None:
    print(f'Resuming from checkpoint: {resume_dir}')
    load_model(resume_dir)

## Visualization helpers
*new* -- live training visualization: a loss-curve plot updated every epoch (train / seen-val / unseen-val total loss, all on the same, comparable unweighted-MSE scale), plus a grid of sample validation images with the model's predicted property values next to the true ones, shown every `image_log_every` epochs (plus the run's final epoch) so you can visually sanity-check predictions -- including each property's % relative error, since raw MSE totals are hard to judge by eye. Notebook-only (uses `IPython.display`); `train_from_scratch.py` does not include this.

In [ ]:
PROPERTY_NAMES = ["YS", "UTS", "EL", "E", "k", "n"]


def unnormalize(img_tensor):
    # inverts transforms.Normalize((0.5), (0.5)): maps roughly [-1, 1] back to [0, 1]
    return (img_tensor * 0.5 + 0.5).clamp(0, 1)


def plot_loss_curves(epochs_so_far, train_hist, seen_hist, unseen_hist):
    # new -- total loss = sum of the 6 per-property MSEs, unweighted (unlike the
    # optimized `loss`, which uses the scale-balancing weights) so train/seen/unseen
    # are on a directly comparable scale.
    plt.figure(figsize=(7, 4))
    plt.plot(epochs_so_far, train_hist, label="Train (unweighted total MSE)")
    plt.plot(epochs_so_far, seen_hist, label="Seen val (total MSE)")
    plt.plot(epochs_so_far, unseen_hist, label="Unseen val (total MSE)")
    plt.xlabel("Epoch")
    plt.ylabel("Total loss (sum of 6 property MSEs)")
    plt.title("Training progress")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def relative_error_pct(pred, true, eps=1e-6):
    # new -- % error, guarded against near-zero true values (none of YS/UTS/EL/E/k/n
    # are ever actually ~0 in this dataset, but this avoids a division blow-up if a
    # future dataset/class table ever has one).
    return 100.0 * (pred - true).abs() / true.abs().clamp(min=eps)


def show_prediction_grid(loader, table, split_name, epoch, n=4):
    # new -- samples n images from `loader`, runs the current EMA model (the same
    # model test() evaluates with), and shows each image next to its predicted vs.
    # true property values and the resulting relative error -- raw MSE totals (in
    # the thousands, since UTS/k are on a large numeric scale) are hard to judge by
    # eye, but "UTS off by 6%" is immediately readable.
    model.eval()
    with torch.no_grad():
        images, labels = next(iter(loader))
        images = aug_transform(images)
        n = min(n, images.size(0))
        mag, ys, uts, el, e, k, n_exp = class_maker(
            batch_size=labels.size(0), labels=labels, tab=table)
        preds = ema_model(images.to(device), mag.long().to(device)).cpu()
        truth = torch.stack([ys, uts, el, e, k, n_exp], dim=1)
        errs = relative_error_pct(preds, truth)

        fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 4))
        if n == 1:
            axes = [axes]
        for i in range(n):
            img = unnormalize(images[i]).squeeze(0).numpy()
            axes[i].imshow(img, cmap="gray")
            axes[i].axis("off")
            caption = "\n".join(
                f"{name}: {preds[i, j]:.2f} / {truth[i, j]:.2f}  ({errs[i, j]:.0f}% err)"
                for j, name in enumerate(PROPERTY_NAMES))
            caption += f"\nmean err: {errs[i].mean():.0f}%"
            axes[i].set_title(caption, fontsize=8, loc="left")
        fig.suptitle(f"{split_name} -- epoch {epoch} (predicted / true, % error)")
        plt.tight_layout()
        plt.show()
    model.train()

## Advanced diagnostics (paper-style)
*new* -- three visualizations modeled on the source paper's own evaluation figures, shown at the same cadence as the prediction grids above (`image_log_every` epochs, plus the run's final epoch):
- **Grad-CAM heatmaps** -- highlight which regions of each micrograph the model actually relied on for its prediction (paper's heatmap figure).
- **Per-class scatter with NRMSE/R²** -- predictions averaged over every image of a class vs. its true value, seen (black) vs. unseen (red) -- the paper's Fig. 8 approach, which reduces per-image microstructure-heterogeneity noise.
- **Ramberg-Osgood stress-strain curves** -- real vs. predicted E/K/n turned into full stress-strain curves for a handful of classes -- the paper's Fig. 9.

These are heavier than the basic loss plot/prediction grid (each involves a backward pass or a full test-set forward pass), so they share the same `image_log_every` cadence rather than running every epoch.

In [ ]:
# new -- Grad-CAM: hooks down3's output ((B,128,64,64), 3 downsampling stages in)
# rather than the deepest bottleneck (bot3, (B,64,8,8)) -- bot3 is coarse enough
# that upsampled back to 512x512 it only ever produces smooth blobs, not detail
# following actual microstructure features. down3 trades some semantic depth for
# 8x finer spatial resolution; pass a different target_layer (e.g. model.bot3) to
# go back to the coarser/deeper view. Uses `model` (not `ema_model`), because
# `ema_model` has requires_grad_(False) on every parameter, so no gradient graph
# is built through it and there is nothing for Grad-CAM to backprop into.
def compute_gradcam(images, mag, target_idx=None, target_layer=None):
    target_layer = target_layer if target_layer is not None else model.down3
    activations = {}
    gradients = {}

    def fwd_hook(module, inp, out):
        activations["value"] = out

    def bwd_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0]

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    was_training = model.training
    model.eval()
    with torch.enable_grad():
        # new -- requires_grad_(True) on the input silences PyTorch's full-backward-
        # hook warning ("no inputs require gradients") and guarantees a gradient
        # graph is built regardless of any surrounding no_grad context.
        images_dev = images.to(device).requires_grad_(True)
        preds = model(images_dev, mag.long().to(device))
        # new -- default target: sum of all 6 predicted properties, i.e. "what did
        # the model look at to produce its predictions overall"; pass target_idx
        # (0..5, matching PROPERTY_NAMES) for a single property's heatmap instead.
        score = preds.sum(dim=1) if target_idx is None else preds[:, target_idx]
        model.zero_grad()
        score.sum().backward()

    h1.remove()
    h2.remove()

    act = activations["value"].detach()
    grad = gradients["value"].detach()
    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * act).sum(dim=1))
    cam_min = cam.amin(dim=(1, 2), keepdim=True)
    cam_max = cam.amax(dim=(1, 2), keepdim=True)
    cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

    # new -- clear the gradients Grad-CAM's backward() just populated, so they
    # can't leak into the next real training step (the training loop already
    # calls optimizer.zero_grad() at the start of every batch, so this is a
    # belt-and-suspenders safeguard, not strictly required).
    model.zero_grad()
    if was_training:
        model.train()
    return cam.detach().cpu(), preds.detach().cpu()


def show_gradcam_grid(loader, table, split_name, epoch, n=4):
    images, labels = next(iter(loader))
    n = min(n, images.size(0))
    images = images[:n]
    labels = labels[:n]
    mag, *_ = class_maker(batch_size=labels.size(0), labels=labels, tab=table)
    cam, _ = compute_gradcam(images, mag)
    cam_up = F.interpolate(
        cam.unsqueeze(1), size=images.shape[-2:], mode="bilinear", align_corners=False
    ).squeeze(1)

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    if n == 1:
        axes = [axes]
    for i in range(n):
        img = unnormalize(images[i]).squeeze(0).numpy()
        axes[i].imshow(img, cmap="gray")
        axes[i].imshow(cam_up[i].numpy(), cmap="jet", alpha=0.45)
        axes[i].axis("off")
    fig.suptitle(f"{split_name} -- epoch {epoch} -- Grad-CAM (down3, all 6 outputs)")
    plt.tight_layout()
    plt.show()


# new -- Fig. 8 style: average this model's predictions over every image that
# belongs to the same class (processing condition), then compare that one
# averaged prediction to the class's single true property vector -- this is what
# the paper calls mitigating "microstructure heterogeneity" (per-image noise) by
# aggregating over all available images of a sample.
def per_class_aggregate(loader, table, label_dict):
    ema_model.eval()
    with torch.no_grad():
        images, labels = next(iter(loader))
        images = aug_transform(images)
        mag, ys, uts, el, e, k, n_exp = class_maker(
            batch_size=labels.size(0), labels=labels, tab=table)
        preds = ema_model(images.to(device), mag.long().to(device)).cpu()
        truth = torch.stack([ys, uts, el, e, k, n_exp], dim=1)

    by_class_preds = {}
    by_class_truth = {}
    for i in range(labels.size(0)):
        c = labels[i].item()
        by_class_preds.setdefault(c, []).append(preds[i])
        by_class_truth[c] = truth[i]

    class_ids = sorted(by_class_preds)
    avg_preds = torch.stack([torch.stack(by_class_preds[c]).mean(dim=0) for c in class_ids])
    class_truth = torch.stack([by_class_truth[c] for c in class_ids])
    class_names = [label_dict[c] for c in class_ids]
    return avg_preds, class_truth, class_names


def nrmse(pred, true):
    # new -- range-normalized RMSE, matching the paper's NRMSE usage in Fig. 8
    rmse = torch.sqrt(torch.mean((pred - true) ** 2))
    return (rmse / (true.max() - true.min())).item()


def r_squared(pred, true):
    ss_res = torch.sum((true - pred) ** 2)
    ss_tot = torch.sum((true - true.mean()) ** 2)
    return (1 - ss_res / ss_tot).item()


def plot_property_scatter(epoch):
    seen_pred, seen_true, _ = per_class_aggregate(seen_test_loader, seen_table, seen_label)
    unseen_pred, unseen_true, _ = per_class_aggregate(unseen_test_loader, unseen_table, unseen_label)

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for j, name in enumerate(PROPERTY_NAMES):
        ax = axes[j // 3, j % 3]
        all_pred = torch.cat([seen_pred[:, j], unseen_pred[:, j]])
        all_true = torch.cat([seen_true[:, j], unseen_true[:, j]])
        lo, hi = all_true.min().item(), all_true.max().item()
        pad = 0.05 * (hi - lo + 1e-6)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="tab:blue", linewidth=1)
        ax.scatter(seen_true[:, j], seen_pred[:, j], color="black", label="Image-out (seen)", s=25)
        ax.scatter(unseen_true[:, j], unseen_pred[:, j], color="red", label="Class-out (unseen)", s=25)
        ax.set_title(f"{name}  NRMSE={nrmse(all_pred, all_true):.3f}  R2={r_squared(all_pred, all_true):.2f}")
        ax.set_xlabel("Target")
        ax.set_ylabel("Prediction")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"Per-class averaged predictions vs. target -- epoch {epoch}")
    plt.tight_layout()
    plt.show()


# new -- Fig. 9 style: the Ramberg-Osgood relation eps(sigma) = sigma/E + (sigma/K)^(1/n).
# E is stored in this dataset's convention as GPa (e.g. ~45), so it's scaled by 1000
# to match K/sigma's MPa scale before dividing.
def ramberg_osgood_curve(E, K, n, sigma_max, num=100):
    # new -- n is physically always positive (strain-hardening exponent, usually
    # ~0.05-0.3 for metals); an undertrained model can still predict a negative or
    # near-zero n, which makes the exponent 1/n a huge (or huge-negative) number and
    # blows (sigma/K)**(1/n) up numerically. Clamp to a small positive floor so the
    # curve stays finite/plottable -- it's a numerical safeguard, not a fix for the
    # underlying bad prediction, so `is_valid` reports whether clamping kicked in.
    is_valid = n > 0.02
    n_safe = max(n, 0.02)
    sigma = torch.linspace(sigma_max / num, sigma_max, num)
    eps = sigma / (E * 1000.0) + (sigma / K).clamp(min=1e-8) ** (1.0 / n_safe)
    # new -- extra safety cap: even with n_safe, a bad K prediction (sigma/K > 1)
    # can still blow up the plastic term. Cap strain at 50% (real AZ80 curves in
    # the paper's reference figure top out around ~10%), so one bad class can't
    # wreck the shared x-axis for every other curve in the plot.
    eps = eps.clamp(max=0.5)
    return eps * 100.0, sigma, is_valid  # strain as %, stress in MPa


def plot_stress_strain_curves(epoch, class_names=None, n_curves=6):
    seen_pred, seen_true, seen_names = per_class_aggregate(seen_test_loader, seen_table, seen_label)
    unseen_pred, unseen_true, unseen_names = per_class_aggregate(unseen_test_loader, unseen_table, unseen_label)

    all_pred = torch.cat([seen_pred, unseen_pred])
    all_true = torch.cat([seen_true, unseen_true])
    all_names = seen_names + unseen_names
    is_unseen = [False] * len(seen_names) + [True] * len(unseen_names)

    if class_names is None:
        # new -- our own default selection: evenly spaced across the sorted class
        # list (mixing seen/unseen) instead of all 46 classes, to keep the plot
        # readable -- pass class_names explicitly to choose specific ones instead.
        idx = torch.linspace(0, len(all_names) - 1, min(n_curves, len(all_names))).long().tolist()
    else:
        idx = [all_names.index(name) for name in class_names]

    fig, ax = plt.subplots(figsize=(7, 6))
    colors = plt.cm.tab10.colors
    for color_i, i in enumerate(idx):
        E_t, K_t, n_t = all_true[i, 3].item(), all_true[i, 4].item(), all_true[i, 5].item()
        E_p, K_p, n_p = all_pred[i, 3].item(), all_pred[i, 4].item(), all_pred[i, 5].item()
        uts_t = all_true[i, 1].item()
        color = colors[color_i % len(colors)]
        eps_t, sig_t, valid_t = ramberg_osgood_curve(E_t, K_t, n_t, sigma_max=uts_t)
        eps_p, sig_p, valid_p = ramberg_osgood_curve(E_p, K_p, n_p, sigma_max=uts_t)
        label = all_names[i] + ("*" if is_unseen[i] else "")
        pred_flag = "" if valid_p else " [invalid n, clamped]"
        ax.plot(eps_t, sig_t, color=color, linestyle="-", label=f"{label} (real)")
        ax.plot(eps_p, sig_p, color=color, linestyle="--", label=f"{label} (pred){pred_flag}")

    ax.set_xlabel("True Strain (%)")
    ax.set_ylabel("True Stress (MPa)")
    ax.set_title(f"Ramberg-Osgood stress-strain curves -- epoch {epoch}\n(* = class-out / unseen)")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Training loop
*ported from train.py* -- loss composition (the deliberate scale-balancing weights `0.223/0.189/17.18/12.32/0.094/675.68*10`), EMA step after every `optimizer.step()`, and the "save best" checkpoint pattern. Uses `training_class_table` (either the real `class_table` or the fallback table built above) instead of `class_table` directly.

*new* -- early stopping on the combined seen+unseen validation loss (`patience` epochs without improvement), added because ~800 epochs from scratch is long enough that it should stop before wasting the remainder of a free Kaggle GPU session.

In [ ]:
epochs_without_improvement = 0

seen_test_results = torch.zeros(n_epoch, 7)
unseen_test_results = torch.zeros(n_epoch, 7)
train_loss_epoch = []

# new -- for the live loss plot (see Visualization helpers above): unweighted
# total train loss (comparable scale to the val totals), plus the epoch numbers
# actually reached, since early stopping can end the run before n_epoch.
epochs_so_far = []
train_total_hist = []
seen_total_hist = []
unseen_total_hist = []

# ported from train.py -- training loop (loss composition, EMA step, checkpointing)
for epoch in range(1, n_epoch + 1):
    # new -- clear previous epoch's printed output/plot before this epoch's own,
    # so the notebook shows only the latest epoch instead of scrolling forever.
    clear_output(wait=True)
    model.train()
    loss_epoch = 0
    loss_ys_total = 0
    loss_uts_total = 0
    loss_el_total = 0
    loss_e_total = 0
    loss_k_total = 0
    loss_n_total = 0

    for i, (images, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        images = aug_transform(images)
        images = images.to(device)
        mag, ys, uts, el, e, k, n = class_maker(batch_size=labels.size(0), labels=labels, tab=training_class_table)
        ys = noise(ys, ys_range)
        uts = noise(uts, uts_range)
        el = noise(el, el_range)
        e = noise(e, e_range)
        k = noise(k, k_range)
        n = noise(n, n_range)

        mag = mag.long().to(device)
        ys = ys.float().to(device)
        uts = uts.float().to(device)
        el = el.float().to(device)
        e = e.float().to(device)
        k = k.float().to(device)
        n = n.float().to(device)

        labels = labels.long().to(device)
        predictions = model(images, mag)

        loss_ys = mse(ys.float(), predictions[:, 0].float())
        loss_uts = mse(uts.float(), predictions[:, 1].float())
        loss_el = mse(el.float(), predictions[:, 2].float())
        loss_e = mse(e.float(), predictions[:, 3].float())
        loss_k = mse(k.float(), predictions[:, 4].float())
        loss_n = mse(n.float(), predictions[:, 5].float())

        loss_ys_total += loss_ys.item()
        loss_uts_total += loss_uts.item()
        loss_el_total += loss_el.item()
        loss_e_total += loss_e.item()
        loss_k_total += loss_k.item()
        loss_n_total += loss_n.item()

        # ported from train.py -- deliberate scale-balancing weights across outputs
        # with very different numeric ranges (not present in the paper text)
        loss = 0.223 * loss_ys + 0.189 * loss_uts + 17.18 * loss_el + 12.32 * loss_e + 0.094 * loss_k + 675.68 * loss_n * 10

        loss.backward()

        optimizer.step()
        ema.step_ema(ema_model, model)
        loss_epoch += loss.item()

    train_loss_epoch.append(loss_epoch / len(train_loader))
    print('Training')
    print('Epoch: [%d/%d]: Loss: %.3f' % (epoch, n_epoch, loss_epoch / len(train_loader)))
    print(f'YS: {loss_ys_total/l} \t UTS: {loss_uts_total/l} \t el: {loss_el_total/l} \t E: {loss_e_total/l} \t k: {loss_k_total/l} \t n: {loss_n_total/l}')

    # new -- unweighted sum of the 6 property MSEs, for the loss plot (directly
    # comparable to seen/unseen val totals, unlike the weighted `loss` above)
    train_total_epoch = (loss_ys_total + loss_uts_total + loss_el_total
                          + loss_e_total + loss_k_total + loss_n_total) / l

    model.eval()
    with torch.no_grad():
        s_test_ys_error, s_test_uts_error, s_test_el_error, s_test_e_error, s_test_k_error, s_test_n_error, s_test_total_error = test(seen_test_loader, seen_table)
        u_test_ys_error, u_test_uts_error, u_test_el_error, u_test_e_error, u_test_k_error, u_test_n_error, u_test_total_error = test(unseen_test_loader, unseen_table)

        seen_test_results[epoch - 1, 0] = s_test_ys_error.item()
        seen_test_results[epoch - 1, 1] = s_test_uts_error.item()
        seen_test_results[epoch - 1, 2] = s_test_el_error.item()
        seen_test_results[epoch - 1, 3] = s_test_e_error.item()
        seen_test_results[epoch - 1, 4] = s_test_k_error.item()
        seen_test_results[epoch - 1, 5] = s_test_n_error.item()
        seen_test_results[epoch - 1, 6] = s_test_total_error.item()

        unseen_test_results[epoch - 1, 0] = u_test_ys_error.item()
        unseen_test_results[epoch - 1, 1] = u_test_uts_error.item()
        unseen_test_results[epoch - 1, 2] = u_test_el_error.item()
        unseen_test_results[epoch - 1, 3] = u_test_e_error.item()
        unseen_test_results[epoch - 1, 4] = u_test_k_error.item()
        unseen_test_results[epoch - 1, 5] = u_test_n_error.item()
        unseen_test_results[epoch - 1, 6] = u_test_total_error.item()

        print('Testing')
        print(f'Seen Loss: {s_test_total_error} \t Unseen Loss: {u_test_total_error}')

        combined_val_loss = (s_test_total_error + u_test_total_error).item()

        # ported from train.py -- "save best" pattern (min_loss comparison / save_model)
        if combined_val_loss < min_loss - min_delta:
            min_loss = combined_val_loss
            epochs_without_improvement = 0
            model_save_dir = os.path.join(save_dir, f'Mic-Mech-Scratch-Best{n_epoch}.pth.tar')
            save_model(model_save_dir)
            print('model saved!')
        else:
            # new -- early stopping on combined seen+unseen validation loss
            epochs_without_improvement += 1
            print(f'No improvement for {epochs_without_improvement}/{patience} epochs')
        print('----------------------')

        # new -- update the live loss plot every epoch, and every `image_log_every`
        # epochs (plus the first and the final epoch of the run -- whether that's
        # n_epoch or an early stop -- so the run's last state is always visible
        # instead of possibly landing between two `image_log_every` checkpoints).
        is_last_epoch = (epoch == n_epoch) or (epochs_without_improvement >= patience)
        epochs_so_far.append(epoch)
        train_total_hist.append(train_total_epoch)
        seen_total_hist.append(s_test_total_error.item())
        unseen_total_hist.append(u_test_total_error.item())
        plot_loss_curves(epochs_so_far, train_total_hist, seen_total_hist, unseen_total_hist)

        if epoch == 1 or epoch % image_log_every == 0 or is_last_epoch:
            show_prediction_grid(seen_test_loader, seen_table, "Seen test", epoch)
            show_prediction_grid(unseen_test_loader, unseen_table, "Unseen test", epoch)
            # new -- paper-style diagnostics (see "Advanced diagnostics" above),
            # shown at the same cadence since they're heavier to compute/render.
            show_gradcam_grid(seen_test_loader, seen_table, "Seen test", epoch)
            show_gradcam_grid(unseen_test_loader, unseen_table, "Unseen test", epoch)
            plot_property_scatter(epoch)
            plot_stress_strain_curves(epoch)

    if epochs_without_improvement >= patience:
        print(f'Early stopping at epoch {epoch} (patience={patience} reached).')
        break

## Save final checkpoint and export CSVs
*ported from train.py* -- CSV export logic (`Train{trial}.csv`, `Seen-Test{trial}.csv`, `Unseen-Test{trial}.csv`), *adjusted (new)* to use `completed_epochs` instead of a hardcoded epoch count, so row counts match the actual training length even if early stopping triggered.

In [ ]:
completed_epochs = epoch  # new -- actual epochs run, may be < n_epoch due to early stopping

model_save_dir = os.path.join(save_dir, f'Mic-Mech-Scratch-Final{n_epoch}.pth.tar')
save_model(model_save_dir)

# ported from train.py -- CSV export, adjusted (new) to use completed_epochs instead of
# a hardcoded epoch count, so rows match actual training length under early stopping
filename = 'Train' + str(trial)
filenamecsv = os.path.join(save_dir, filename + '.csv')
with open(filenamecsv, 'w', newline='') as csvfile:
    fieldnames = ['Epoch', 'Loss']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i in range(1, completed_epochs + 1):
        writer.writerow({'Epoch': i, 'Loss': train_loss_epoch[i - 1]})

filename = 'Seen-Test' + str(trial)
filenamecsv = os.path.join(save_dir, filename + '.csv')
with open(filenamecsv, 'w', newline='') as csvfile:
    fieldnames = ['Epoch', 'YS Loss', 'UTS Loss', 'EL Loss', 'E Loss', 'k Loss', 'n Loss', 'Total Loss']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i in range(1, completed_epochs + 1):
        writer.writerow({'Epoch': i, 'YS Loss': seen_test_results[i - 1, 0].item(), 'UTS Loss': seen_test_results[i - 1, 1].item(), 'EL Loss': seen_test_results[i - 1, 2].item(), 'E Loss': seen_test_results[i - 1, 3].item(), 'k Loss': seen_test_results[i - 1, 4].item(), 'n Loss': seen_test_results[i - 1, 5].item(), 'Total Loss': seen_test_results[i - 1, 6].item()})

filename = 'Unseen-Test' + str(trial)
filenamecsv = os.path.join(save_dir, filename + '.csv')
with open(filenamecsv, 'w', newline='') as csvfile:
    fieldnames = ['Epoch', 'YS Loss', 'UTS Loss', 'EL Loss', 'E Loss', 'k Loss', 'n Loss', 'Total Loss']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i in range(1, completed_epochs + 1):
        writer.writerow({'Epoch': i, 'YS Loss': unseen_test_results[i - 1, 0].item(), 'UTS Loss': unseen_test_results[i - 1, 1].item(), 'EL Loss': unseen_test_results[i - 1, 2].item(), 'E Loss': unseen_test_results[i - 1, 3].item(), 'k Loss': unseen_test_results[i - 1, 4].item(), 'n Loss': unseen_test_results[i - 1, 5].item(), 'Total Loss': unseen_test_results[i - 1, 6].item()})

if device.type == 'cuda':
    print(torch.cuda.memory_summary(device=device))